# Comparative genomics of truncated EAAT isoforms — reproducible analysis

Companion notebook to **Karagöl, Karagöl & Zhang**, *Multi-level Comparative Genomic Analysis
Reveals Evolutionary Influence of Truncated Isoforms in Glutamate Transporter Genes*.

Running this notebook top to bottom reproduces every new number in the revised manuscript:
Tables 4, 5, 6, Tables S5–S9, and Figures 4, 5 and 6.

**What it does**

| § | Analysis | Produces |
|---|----------|----------|
| 1 | Parse and validate the TBLASTN exports | Table S5 |
| 2 | Demonstrate the ascertainment bias in hit-level scores | (diagnostic) |
| 3 | Best-hit molecular clock and ANCOVA | Tables 6, S7, S8 |
| 4 | ConSurf profiles and alignment-based mapping | Tables 5, S9 |
| 5 | Domain-resolved structural superposition | Table S6 |
| 6 | Compensatory base-pair classification | Table 4 |
| 7 | Figures | Figures 4–6 |
| 8 | Audit of missing datasets | (checklist) |

**What it does not do.** No molecular dynamics simulation is run, and no simulated
observable is produced anywhere. The SpliceAI clustering analysis is not included because
the underlying Δ-score tables are not in the repository — §8 checks for them and will tell
you if that has changed.

---

### Parameters you may want to change

* `DIVERGENCE_MYA` (§1) — divergence times, currently from TimeTree.
* `SCAFFOLD` / `TRANSPORT` (§4) — EAAT domain boundaries in P43004 numbering. These are a
  structural judgement; the notebook reports how sensitive the result is to them.
* `MIN_QUERY_COVERAGE` (§3) — the coverage floor for calling a best reciprocal-region hit.
* `MIN_PAIR_SUPPORT` (§6) — how many of the six aligned sequences must support a base pair.

## 0. Setup

In [ ]:
import importlib, subprocess, sys

REQUIRED = {'pandas':'pandas', 'numpy':'numpy', 'scipy':'scipy',
            'statsmodels':'statsmodels', 'Bio':'biopython', 'matplotlib':'matplotlib'}
missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('installing:', ' '.join(missing))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
else:
    print('all dependencies present')

In [ ]:
import os, re, glob, shutil, subprocess, warnings, zipfile
import numpy as np, pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200, 'display.max_columns', 40)

# ---- where the repository lives -------------------------------------------
# Leave REPO = None to clone it; or set it to an existing local checkout.
REPO = None
REPO_URL = 'https://github.com/karagol-alper/EAATisoformgenomics.git'

if REPO is None:
    REPO = os.path.abspath('EAATisoformgenomics')
    if not os.path.isdir(REPO):
        print('cloning', REPO_URL)
        subprocess.run(['git','clone','--depth','1',REPO_URL,REPO], check=True)
    else:
        print('using existing clone')
assert os.path.isdir(REPO), REPO
OUT = os.path.abspath('outputs'); os.makedirs(OUT, exist_ok=True)
print('repository :', REPO)
print('outputs    :', OUT)

## 1. Parse and validate the TBLASTN exports

The repository ships 105 result tables — 15 genome assemblies × 7 *SLC1A2* queries
(canonical P43004 plus six truncated isoforms) — inside `TBLASTNresults.zip`.

One export (*Theropithecus gelada*, A0A2R8Y4N0) was written without the query-name column.
It is recovered by elimination against the other six gelada queries and by its maximum query
coordinate, 330 residues, which matches that query length uniquely.

The validation block at the end is the important part: it checks the parse against the
**published Figures S17–S19**. If those numbers stop matching, something upstream changed.

In [ ]:
ZIP = os.path.join(REPO, 'Evolutionary Analysis', 'TBLASTN', 'TBLASTNresults.zip')
WORK = os.path.abspath('_tblastn'); os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(WORK)
SRC = glob.glob(os.path.join(WORK, '**', 'blast_results_2-*.csv'), recursive=True)
print(len(SRC), 'export tables found')

QLEN = {'P43004':574, 'C9J9N5':409, 'A0A2R8Y4D1':397, 'A0A2R8Y642':435,
        'A0A2R8Y4N0':330, 'A0A2R8YHI4':155, 'H0YEB1':91}
ARCH = {'P43004':'canonical', 'C9J9N5':'scaffold', 'A0A2R8Y4D1':'scaffold',
        'A0A2R8Y642':'scaffold', 'A0A2R8Y4N0':'scaffold',
        'A0A2R8YHI4':'cytosolic', 'H0YEB1':'cytosolic'}
TM   = {'P43004':8, 'C9J9N5':7, 'A0A2R8Y4D1':7, 'A0A2R8Y642':4,
        'A0A2R8Y4N0':5, 'A0A2R8YHI4':0, 'H0YEB1':0}

# clade and divergence time from Homo sapiens (TimeTree; Kumar et al. 2017)
DIVERGENCE_MYA = {
 'Homo_sapiens':('Hominidae',0.0),      'Pan_troglodytes':('Hominidae',6.4),
 'Pan_paniscus':('Hominidae',6.4),      'Gorilla_gorilla':('Hominidae',8.6),
 'Pongo_abelii':('Hominidae',15.2),     'Nomascus_leucogenys':('Hominidae',19.5),
 'Macaca_mulatta':('Primates_nh',28.8), 'Theropithecus_gelada':('Primates_nh',28.8),
 'Otolemur_garnettii':('Primates_nh',69.0),
 'Mus_musculus':('Outgroup',87.0),      'Rattus_norvegicus':('Outgroup',87.0),
 'Sus_scrofa':('Outgroup',94.0),        'Ovis_aries_rambouillet':('Outgroup',94.0),
 'Gallus_gallus':('Outgroup',319.0),    'Caenorhabditis_elegans':('Outgroup',736.0)}

In [ ]:
records, recovered = [], []
for path in sorted(SRC):
    fn = os.path.basename(path)
    sp = fn.split('ResultsTable-')[1].split('_Tools_Blast_')[0]
    df = pd.read_csv(path, quotechar='"', engine='python', on_bad_lines='skip')
    df.columns = [c.strip() for c in df.columns]
    if 'Query name' in df.columns:
        df['query'] = df['Query name'].astype(str).str.extract(r'^(?:sp|tr)\|([A-Z0-9]+)\|')[0]
    else:
        # malformed export: identify by which query is missing for this species
        df['query'] = None; df['recovered'] = True; recovered.append((sp, fn, len(df)))
    df['species'] = sp
    if 'recovered' not in df: df['recovered'] = False
    records.append(df[['query','species','recovered','Score','E-val','%ID','Length',
                       'Query start','Query end']])

d = pd.concat(records, ignore_index=True)

# resolve the malformed export(s)
for sp, fn, n in recovered:
    present = set(d[(d.species==sp) & d['query'].notna()]['query'])
    absent  = sorted(set(QLEN) - present)
    blk = (d.species==sp) & d['query'].isna()
    qmax = pd.to_numeric(d.loc[blk,'Query end'], errors='coerce').max()
    cand = [q for q in absent if abs(QLEN[q]-qmax) <= 1]
    assert len(cand)==1, f'cannot resolve {fn}: absent={absent} max_query_end={qmax}'
    d.loc[blk, 'query'] = cand[0]
    print(f'recovered {fn}: {n} rows -> {cand[0]} (max query end {qmax:.0f}, length {QLEN[cand[0]]})')

d = d[d['query'].isin(QLEN)].copy()
d['score']  = pd.to_numeric(d['Score'], errors='coerce')
d['pid']    = pd.to_numeric(d['%ID'].astype(str).str.extract(r'([\d.]+)')[0], errors='coerce')
d['qs']     = pd.to_numeric(d['Query start'], errors='coerce')
d['qe']     = pd.to_numeric(d['Query end'],   errors='coerce')
d = d.dropna(subset=['score','pid'])
d['qlen']  = d['query'].map(QLEN)
d['arch']  = d['query'].map(ARCH)
d['clade'] = d['species'].map(lambda s: DIVERGENCE_MYA[s][0])
d['mya']   = d['species'].map(lambda s: DIVERGENCE_MYA[s][1])
d['alnfrac'] = (d.qe - d.qs + 1) / d.qlen
d['score_per_res'] = d.score / d.qlen
print(f'\n{len(d)} alignments parsed')

In [ ]:
# ---- validation against the published Figures S17-S19 ----------------------
PUBLISHED = {'hits':   {'P43004':457,'C9J9N5':447,'A0A2R8Y4D1':443,'A0A2R8Y4N0':399,
                        'A0A2R8Y642':706,'A0A2R8YHI4':385,'H0YEB1':260},
             'sum':    {'P43004':208900,'C9J9N5':143901.3,'A0A2R8Y4D1':135482.4,
                        'A0A2R8Y4N0':136284.8,'A0A2R8Y642':168079.3,
                        'A0A2R8YHI4':60317.6,'H0YEB1':26256.5},
             'median': {'P43004':379,'C9J9N5':264,'A0A2R8Y4D1':240,'A0A2R8Y4N0':313,
                        'A0A2R8Y642':217,'A0A2R8YHI4':115,'H0YEB1':46.4}}

# Figures S17-S19 were produced from the 104 well-formed exports, so the
# published-equivalent subset excludes the rows recovered above.
d_pub = d[~d.recovered]
g = d_pub.groupby('query')
chk = pd.DataFrame({'hits':g.size(), 'sum_score':g.score.sum().round(1),
                    'median_score':g.score.median()})
chk['pub_hits']   = chk.index.map(PUBLISHED['hits'])
chk['pub_sum']    = chk.index.map(PUBLISHED['sum'])
chk['pub_median'] = chk.index.map(PUBLISHED['median'])
chk['hits_match']   = chk.hits == chk.pub_hits
chk['median_match'] = np.isclose(chk.median_score, chk.pub_median, atol=0.05)
display(chk[['hits','pub_hits','hits_match','sum_score','pub_sum',
             'median_score','pub_median','median_match']])
n_ok = int(chk.hits_match.sum())
print(f'\n{n_ok}/7 queries reproduce the published hit total exactly '
      f'(n = {len(d_pub)} alignments from the well-formed exports).')
if n_ok < 7:
    for q in chk.index[~chk.hits_match]:
        print(f'  mismatch: {q}  re-parsed {chk.loc[q,"hits"]}  vs published {chk.loc[q,"pub_hits"]}')
    print('\nKnown discrepancy: A0A2R8Y642 re-parses to 664 against the published 706.')
    print('The outgroup stratum agrees exactly (236), so the difference lies entirely')
    print('within the primate strata. See SI Addendum, Section B.')

### Table S5 — cross-clade summary

In [ ]:
rows = []
for q, s in d_pub.groupby('query'):
    r = {'isoform':q, 'TM':TM[q], 'architecture':ARCH[q], 'length':QLEN[q], 'total_hits':len(s)}
    for cl, lab in [('Hominidae','hominid'), ('Primates_nh','primates_nh'), ('Outgroup','outgroup')]:
        sub = s[s.clade==cl]
        r[f'n_{lab}'] = len(sub)
        r[f'medpid_{lab}'] = round(sub.pid.median(), 2) if len(sub) else np.nan
    rows.append(r)
table_S5 = pd.DataFrame(rows).sort_values('TM', ascending=False).reset_index(drop=True)
table_S5.to_csv(f'{OUT}/table_S5.csv', index=False)
display(table_S5)

## 2. Why hit-level scores cannot be used

The obvious analysis — regress normalised alignment score on divergence time across all
alignments — does not work, and it is worth seeing why before moving on.

TBLASTN only reports alignments that clear the expectation-value threshold. At greater
phylogenetic distance the weakly conserved segments of a query stop clearing it, so the
surviving alignments are an increasingly selected subset. The score distribution among
*detected* hits is therefore conditioned on detection and stays flat, or rises, with
divergence. This is survivorship bias.

The cell below fits that model anyway. Near-zero R² and inconsistent slope signs are the
diagnostic.

In [ ]:
diag = []
for q, s in d.groupby('query'):
    s = s[s.score_per_res > 0]
    m = sm.OLS(np.log10(s.score_per_res.values),
               sm.add_constant(s.mya.values)).fit(cov_type='HC3')
    diag.append({'isoform':q, 'n_alignments':len(s), 'slope':round(m.params[1], 6),
                 'sign':'neg' if m.params[1] < 0 else 'POS (wrong direction)',
                 'R2':round(m.rsquared, 4), 'p':f'{m.pvalues[1]:.3f}'})
diag = pd.DataFrame(diag)
display(diag)
print(f'R2 range: {diag.R2.min():.4f} to {diag.R2.max():.4f}')
print(f"{(diag.sign.str.startswith('POS')).sum()}/7 queries show the wrong sign.")
print('\nThe model is uninformative. Section 3 uses one observation per species instead.')

## 3. Best-hit molecular clock

For each query–species pair, keep the highest-scoring alignment spanning at least
`MIN_QUERY_COVERAGE` of the query, and drop human self-hits. That gives 14 independent
observations per query.

Divergence (100 − %ID) is regressed on log₁₀ divergence time. The architecture comparison
is an ANCOVA with an interaction term and standard errors clustered on species, since the
same 14 genomes are reused for every query.

In [ ]:
MIN_QUERY_COVERAGE = 0.50

best = (d[d.alnfrac >= MIN_QUERY_COVERAGE]
          .sort_values('score', ascending=False)
          .groupby(['query','species'], as_index=False).first())
best = best[best.species != 'Homo_sapiens'].copy()
best['logt'] = np.log10(best.mya)
best['divg'] = 100 - best.pid
print(f'{len(best)} observations ({best.species.nunique()} species x {best["query"].nunique()} queries)')

ORDER = ['P43004','C9J9N5','A0A2R8Y4D1','A0A2R8Y4N0','A0A2R8Y642','A0A2R8YHI4','H0YEB1']
rows = []
for q in ORDER:
    s = best[best['query']==q]
    m = sm.OLS(s.divg.values, sm.add_constant(s.logt.values)).fit()
    rho, p_rho = stats.spearmanr(s.mya, s.pid)
    rows.append({'isoform':q, 'architecture':ARCH[q], 'n':len(s),
      'pid_rodent': round(s[s.species.isin(['Mus_musculus','Rattus_norvegicus'])].pid.mean(),2),
      'pid_chicken':round(s[s.species=='Gallus_gallus'].pid.mean(),2),
      'pid_celegans':round(s[s.species=='Caenorhabditis_elegans'].pid.mean(),2),
      'slope':round(m.params[1],3), 'SE':round(m.bse[1],3),
      'p':round(m.pvalues[1],4), 'R2':round(m.rsquared,3), 'rho':round(rho,3)})
table_6 = pd.DataFrame(rows)
table_6.to_csv(f'{OUT}/table_6.csv', index=False)
display(table_6)

**ANCOVA (Table S7).** The result to look at is the interaction term. A significant main
effect of architecture with a null interaction means the two classes decay on *parallel*
trajectories separated by a constant offset — the constraint differential was set early and
has not drifted.

In [ ]:
iso = best[best['query'] != 'P43004'].copy()
anc = smf.ols("divg ~ logt * C(arch, Treatment('scaffold'))", data=iso).fit(
          cov_type='cluster', cov_kwds={'groups': iso['species']})
print(anc.summary().tables[1])
print(f'n = {int(anc.nobs)}, clusters = {iso.species.nunique()}, R2 = {anc.rsquared:.3f}')

main = anc.params.filter(like='T.cytosolic]').iloc[0]
inter = anc.params.filter(like='logt:').iloc[0]
p_inter = anc.pvalues.filter(like='logt:').iloc[0]
print(f'\nArchitecture offset : {main:+.3f} pp')
print(f'Interaction         : {inter:+.3f} pp per log10 Myr, p = {p_inter:.3f}')
print('->', 'PARALLEL trajectories (constant offset)' if p_inter > .05
      else 'DIVERGING trajectories - interpretation in the manuscript would need revision')

**Paired contrasts against the canonical transporter, and Table S8.**

In [ ]:
piv = best.pivot_table(index='species', columns='query', values='pid')
con = []
for q in ORDER[1:]:
    s = piv[['P43004', q]].dropna()
    W, p = stats.wilcoxon(s[q], s['P43004'])
    con.append({'isoform':q, 'architecture':ARCH[q], 'n':len(s),
                'mean_delta':round((s[q]-s['P43004']).mean(),2),
                'median_delta':round((s[q]-s['P43004']).median(),2),
                'W':W, 'p':round(p,4),
                'direction':'more conserved' if (s[q]-s['P43004']).mean()>0 else 'less conserved'})
display(pd.DataFrame(con))

sp_order = [s for s,_ in sorted(DIVERGENCE_MYA.items(), key=lambda kv: kv[1][1])
            if s != 'Homo_sapiens']
table_S8 = piv.reindex(sp_order)[ORDER].round(2)
table_S8.to_csv(f'{OUT}/table_S8.csv')
display(table_S8)

## 4. ConSurf profiles and alignment-based mapping

Two things matter here.

**Domain boundaries.** The EAAT fold splits into a scaffold (trimerisation) domain and a
transport domain. The boundaries below are in P43004 numbering, from the archaeal and human
transporter structures. They are a structural judgement — edit them and re-run; the
sensitivity check at the end of this section reports how much the conclusion moves.

**Register.** Four of the six isoforms are *not* positionally colinear with the canonical
protein. Comparing them by residue index silently misaligns them — A0A2R8Y642 returns
r = −0.157 that way and looks like a non-mimic, when in fact it matches at r = 0.877 once
its 50-residue offset is applied. Mapping is therefore done by global pairwise alignment.

In [ ]:
SCAFFOLD  = {'TM1':(41,73), 'TM2':(93,124), 'TM4a-c':(157,232), 'TM5':(238,270)}
TRANSPORT = {'TM3':(128,152), 'TM6':(277,300), 'HP1':(301,330),
             'TM7':(331,367), 'HP2':(368,399), 'TM8':(440,470)}

SEVO = os.path.join(REPO, 'Structural Evolution', '2-EAA2')
NAMES = ['EAA2','C9J9N5','A0A2R8Y4D1','A0A2R8Y642','A0A2R8Y4N0','A0A2R8YHI4','H0YEB1']

def read_grades(path):
    rows = []
    for line in open(path, errors='ignore'):
        if '\t' not in line: continue
        f = [c.strip() for c in line.split('\t')]
        if len(f) < 9 or not f[0].isdigit(): continue
        try:
            rows.append({'pos':int(f[0]), 'aa':f[1], 'score':float(f[3]),
                         'grade':int(re.sub(r'\D','',f[4])), 'lowconf':'*' in f[4],
                         'burial':f[6]})
        except (ValueError, IndexError):
            continue
    return pd.DataFrame(rows)

G = {}
for k in NAMES:
    hits = glob.glob(os.path.join(SEVO, k, '*consurf_grades.txt'))
    if hits: G[k] = read_grades(hits[0])
    else: print('missing grades:', k)

summary = pd.DataFrame([{'protein':k, 'n_res':len(v),
    'median':round(v.score.median(),3), 'pct_grade_8_9':round(100*(v.grade>=8).mean(),1),
    'pct_buried':round(100*(v.burial=='b').mean(),1)} for k,v in G.items()])
display(summary)

### Table 5 — conservation-profile correspondence

In [ ]:
from Bio import Align
from Bio.Align import substitution_matrices

SEQ = {k: ''.join(v.sort_values('pos').aa) for k, v in G.items()}
aligner = Align.PairwiseAligner(mode='global')
aligner.substitution_matrix = substitution_matrices.load('BLOSUM62')
aligner.open_gap_score, aligner.extend_gap_score = -11, -1

MAP, rows = {}, []
for k in NAMES[1:]:
    aln = aligner.align(SEQ['EAA2'], SEQ[k])[0]
    pairs = [(cs+o+1, is_+o+1)
             for (cs,ce),(is_,ie) in zip(*aln.aligned) for o in range(ce-cs)]
    MAP[k] = pairs
    dc, di = G['EAA2'].set_index('pos'), G[k].set_index('pos')
    ok = [(a,b) for a,b in pairs if a in dc.index and b in di.index
          and dc.loc[a,'aa'] == di.loc[b,'aa']]
    sc = np.array([dc.loc[a,'score'] for a,_ in ok])
    si = np.array([di.loc[b,'score'] for _,b in ok])
    r, pr = stats.pearsonr(si, sc); rho, _ = stats.spearmanr(si, sc)
    offs = {a-b for a,b in ok}
    rows.append({'isoform':k, 'matched':len(ok), 'pct_of_isoform':round(100*len(ok)/len(di),1),
                 'register_offset': min(offs) if len(offs)<=2 else 'variable',
                 'pearson_r':round(r,3), 'spearman':round(rho,3),
                 'mean_shift':round((si-sc).mean(),3),
                 'RMS_delta':round(np.sqrt(((si-sc)**2).mean()),3)})
table_5 = pd.DataFrame(rows)
table_5.to_csv(f'{OUT}/table_5.csv', index=False)
display(table_5)

# what naive positional comparison would have given, for comparison
naive = G['A0A2R8Y642'][['pos','aa','score']].merge(
        G['EAA2'][['pos','aa','score']], on='pos', suffixes=('_i','_c'))
naive = naive[naive.aa_i == naive.aa_c]
print(f'\nA0A2R8Y642 by residue index: n={len(naive)}, '
      f'r={stats.pearsonr(naive.score_i, naive.score_c)[0]:+.3f}')
print(f'A0A2R8Y642 after alignment : n={table_5.loc[table_5.isoform=="A0A2R8Y642","matched"].iloc[0]}, '
      f'r={table_5.loc[table_5.isoform=="A0A2R8Y642","pearson_r"].iloc[0]:+.3f}')

### Domain contrast and Table S9

In [ ]:
def in_seg(p, segs): return any(a <= p <= b for a, b in segs.values())

g = G['EAA2']
sc_r = g[[in_seg(p, SCAFFOLD)  for p in g.pos]]
tr_r = g[[in_seg(p, TRANSPORT) for p in g.pos]]
oth  = g[~g.pos.isin(sc_r.pos) & ~g.pos.isin(tr_r.pos)]
for lab, s in [('scaffold',sc_r), ('transport',tr_r), ('loops/termini',oth)]:
    print(f'{lab:15s} n={len(s):4d}  median={s.score.median():+.3f}  '
          f'grade>=8 {100*(s.grade>=8).mean():5.1f}%  buried {100*(s.burial=="b").mean():5.1f}%')
U, p_dom = stats.mannwhitneyu(sc_r.score, tr_r.score)
print(f'\nscaffold vs transport: U={U:.0f}, p={p_dom:.3e}')
print('->', 'TRANSPORT domain more conserved' if tr_r.score.median() < sc_r.score.median()
      else 'SCAFFOLD more conserved')

rows = []
for k in NAMES:
    di = G[k].set_index('pos')
    pairs = [(p,p) for p in di.index] if k=='EAA2' else MAP.get(k, [])
    s = [di.loc[b,'score'] for a,b in pairs if b in di.index and in_seg(a, SCAFFOLD)]
    t = [di.loc[b,'score'] for a,b in pairs if b in di.index and in_seg(a, TRANSPORT)]
    if len(s) < 10 or len(t) < 10: continue
    rows.append({'protein':k, 'n_scaffold':len(s), 'n_transport':len(t),
                 'med_scaffold':round(np.median(s),3), 'med_transport':round(np.median(t),3),
                 'difference':round(np.median(s)-np.median(t),3),
                 'p':f'{stats.mannwhitneyu(s,t)[1]:.2e}'})
table_S9 = pd.DataFrame(rows)
table_S9.to_csv(f'{OUT}/table_S9.csv', index=False)
display(table_S9)

**Sensitivity of the domain result to the boundary definition.** Each boundary is jittered
by ±10 residues 200 times; the cell reports how often the transport domain still comes out
as the more conserved. If this is not close to 100%, the boundaries are doing the work and
the claim should be weakened.

In [ ]:
rng = np.random.default_rng(0)
wins, ps = 0, []
for _ in range(200):
    S = {k:(a+rng.integers(-10,11), b+rng.integers(-10,11)) for k,(a,b) in SCAFFOLD.items()}
    T = {k:(a+rng.integers(-10,11), b+rng.integers(-10,11)) for k,(a,b) in TRANSPORT.items()}
    s = g[[in_seg(p,S) for p in g.pos]].score
    t = g[[in_seg(p,T) for p in g.pos]].score
    ps.append(stats.mannwhitneyu(s,t)[1])
    wins += (t.median() < s.median())
print(f'transport more conserved in {wins}/200 jittered boundary sets ({100*wins/200:.1f}%)')
print(f'median p across jitters: {np.median(ps):.2e}   worst p: {max(ps):.2e}')

## 5. Domain-resolved structural superposition (Table S6)

Isoform AlphaFold2 models are superimposed on the canonical model over aligned identical
residues, globally and per domain. **No outlier rejection is applied**, so these values are
larger than the core similarities of Table S3, which used five rejection cycles. The two
measures answer different questions: Table S3 asks how closely the conserved cores agree,
this asks how much of each isoform departs from the canonical fold.

In [ ]:
from Bio.PDB import PDBParser, Superimposer
parser = PDBParser(QUIET=True)

def ca_atoms(pdb):
    st = parser.get_structure('x', pdb)
    return {r.id[1]: r['CA'] for r in st[0].get_residues() if 'CA' in r}

can_pdb = glob.glob(os.path.join(SEVO,'EAA2','*With_Conservation_Scores.pdb'))[0]
can = ca_atoms(can_pdb)

rows = []
for k in ['C9J9N5','A0A2R8Y4D1','A0A2R8Y642','A0A2R8Y4N0']:
    f = glob.glob(os.path.join(SEVO,k,'*With_Conservation_Scores.pdb'))
    if not f or k not in MAP: continue
    iso = ca_atoms(f[0])
    prs = [(a,b) for a,b in MAP[k] if a in can and b in iso]
    def rms(sel):
        if len(sel) < 4: return np.nan, len(sel)
        sup = Superimposer()
        sup.set_atoms([can[a] for a,_ in sel], [iso[b] for _,b in sel])
        return sup.rms, len(sel)
    r_all, n_all = rms(prs)
    r_s, n_s = rms([(a,b) for a,b in prs if in_seg(a, SCAFFOLD)])
    r_t, n_t = rms([(a,b) for a,b in prs if in_seg(a, TRANSPORT)])
    rows.append({'isoform':k, 'aligned_CA':n_all, 'RMSD_all':round(r_all,3),
                 'n_scaffold':n_s, 'RMSD_scaffold':round(r_s,3),
                 'n_transport':n_t, 'RMSD_transport':round(r_t,3)})
table_S6 = pd.DataFrame(rows)
table_S6.to_csv(f'{OUT}/table_S6.csv', index=False)
display(table_S6)
print('Note: A0A2R8Y4N0 retains too few scaffold residues for that column to be interpreted.')

## 6. Compensatory base-pair substitution (Table 4)

RNAalifold reports, for each predicted base pair, how the pairing nucleotides vary across the
six aligned hominid sequences. Pairs are classified against the majority type:

* **invariant** — one base-pair type only;
* **consistent** — a variant differs at one position but still pairs;
* **compensatory** — a variant differs at *both* positions, a double substitution that
  restores pairing. This is the signature the negative covariance term is picking up.

Covariation tables are deposited for EAA1 and EAA5 only. The files for EAA2, EAA3 and EAA4
contain the consensus structure line without a pair list, so those genes cannot be analysed
until they are re-run with pair output enabled.

In [ ]:
MIN_PAIR_SUPPORT = 3   # of 6 aligned sequences

def parse_alifold(path):
    rows = []
    for line in open(path, errors='ignore'):
        m = re.match(r'\s*(\d+)\s+(\d+)\s+(\d+)\s+([\d.]+)%\s+([\d.]+)\s+(.*)', line)
        if not m: continue
        bps = dict((b.split(':')[0], int(b.split(':')[1]))
                   for b in m.group(6).split() if ':' in b)
        rows.append({'i':int(m.group(1)), 'j':int(m.group(2)),
                     'prob':float(m.group(4)), 'entropy':float(m.group(5)), 'bps':bps})
    return pd.DataFrame(rows)

def classify(bps):
    types = [t for t in bps if len(t)==2]
    if len(types) <= 1: return 'invariant'
    maj = max(bps, key=bps.get)
    comp = cons = False
    for t in types:
        if t == maj: continue
        diff = (t[0]!=maj[0]) + (t[1]!=maj[1])
        if diff == 2: comp = True
        elif diff == 1: cons = True
    return 'compensatory' if comp else ('consistent' if cons else 'invariant')

GENES = {'EAA1':('1-EAA1',4473), 'EAA2':('2-EAA2',12010), 'EAA3':('3-EAA3',3905),
         'EAA4':('4-EAA4',2947), 'EAA5':('5-EAA5',2909)}
COV, rows = {}, []
for gene,(folder,L) in GENES.items():
    p = os.path.join(REPO,'RNA','Consensus',folder,'alifold.out')
    df = parse_alifold(p) if os.path.exists(p) else pd.DataFrame()
    if len(df) == 0:
        print(f'{gene}: no pair list in alifold.out - skipped'); continue
    df['cls'] = df.bps.map(classify)
    hi = df[df.prob >= 100*MIN_PAIR_SUPPORT/6].copy()
    hi['span'] = hi.j - hi.i
    COV[gene] = (hi, L)
    vc = hi.cls.value_counts()
    rows.append({'gene':gene, 'supported_pairs':len(hi),
        'invariant':f"{vc.get('invariant',0)} ({100*vc.get('invariant',0)/len(hi):.1f}%)",
        'consistent':f"{vc.get('consistent',0)} ({100*vc.get('consistent',0)/len(hi):.1f}%)",
        'compensatory':f"{vc.get('compensatory',0)} ({100*vc.get('compensatory',0)/len(hi):.1f}%)",
        'mean_span_invariant':round(hi[hi.cls=='invariant'].span.mean(),1),
        'mean_span_compensatory':round(hi[hi.cls=='compensatory'].span.mean(),1)})
table_4 = pd.DataFrame(rows)
table_4.to_csv(f'{OUT}/table_4.csv', index=False)
display(table_4)

for gene,(hi,L) in COV.items():
    inv  = hi[hi.cls=='invariant']
    comp = hi[hi.cls=='compensatory']
    if len(comp) <= 5:
        print(f'{gene}: only {len(comp)} compensatory pairs - not tested'); continue
    U, p_span = stats.mannwhitneyu(inv.span, comp.span)
    print(f'{gene}:  span  invariant {inv.span.mean():7.1f} nt  vs  compensatory '
          f'{comp.span.mean():7.1f} nt   (U={U:.0f}, p={p_span:.2e})')
    print(f'{gene}:  entropy  invariant {inv.entropy.mean():.3f}  vs  compensatory '
          f'{comp.entropy.mean():.3f}')

In [ ]:
# positional density in 100-nt windows
WIN = 100
DENS = {}
for gene,(hi,L) in COV.items():
    nb = L//WIN + 1
    all_d, comp_d = np.zeros(nb), np.zeros(nb)
    for _, r in hi.iterrows():
        for pos in (r.i, r.j):
            bidx = min(int(pos)//WIN, nb-1)
            all_d[bidx] += 1
            if r.cls == 'compensatory': comp_d[bidx] += 1
    DENS[gene] = {'all':all_d, 'comp':comp_d, 'L':L, 'W':WIN}
    frac = np.divide(comp_d, all_d, out=np.zeros(nb), where=all_d>0)
    hot = [(i*WIN, (i+1)*WIN) for i in np.where((frac>0.9) & (all_d>=10))[0]]
    print(f'{gene}: windows saturated with compensatory pairs -> {hot if hot else "none"}')

## 7. Figures 4–6

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':8,'axes.linewidth':.7,
    'xtick.major.width':.7,'ytick.major.width':.7,'axes.labelsize':8.2,
    'legend.fontsize':7.2,'savefig.dpi':600})
W_IN = 6.8
CAN, SCA, CYT, GREY = '#16324F', '#2E8B77', '#C05B5B', '#9BA6B2'
SCAF_C, TRAN_C = '#4A7FB5', '#D9713C'
COLOR = {q: (CAN if ARCH[q]=='canonical' else SCA if ARCH[q]=='scaffold' else CYT) for q in ORDER}
def tidy(ax):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
def plab(ax, s, x=-.16, y=1.06):
    ax.text(x, y, s, transform=ax.transAxes, fontsize=10.5, fontweight='bold', va='bottom')
def fmt_p(p):
    """Typeset a p-value as m x 10^e rather than the default 7.9e-10."""
    m, e = f'{p:.1e}'.split('e')
    return f'$p$ = {m} \u00d7 10$^{{{int(e)}}}$'

In [ ]:
fig = plt.figure(figsize=(W_IN, 2.55))
gs = fig.add_gridspec(1,3, width_ratios=[1.25,1,1], wspace=.42,
                      left=.075, right=.995, top=.86, bottom=.24)
ax = fig.add_subplot(gs[0]); tidy(ax)
for sp, txt in [('Mus_musculus','rodents'), ('Gallus_gallus','chicken'),
                ('Caenorhabditis_elegans','$C.\\ elegans$')]:
    xv = DIVERGENCE_MYA[sp][1]
    ax.axvline(xv, color=GREY, ls=':', lw=.55, zorder=0)
    ax.text(xv*1.05, 102, txt, rotation=90, va='top', ha='left', fontsize=6, color='#7A8591')
for q in ORDER:
    s = best[best['query']==q].sort_values('mya')
    lw = 1.5 if q=='P43004' else .85
    ax.plot(s.mya, s.pid, '-', color=COLOR[q], lw=lw, alpha=1 if q=='P43004' else .8)
    ax.plot(s.mya, s.pid, 'o', color=COLOR[q], ms=2.6, mec='white', mew=.4)
ax.set_xscale('log'); ax.set_xlim(4,1100); ax.set_ylim(28,103)
ax.set_xlabel('Divergence from human (Myr)'); ax.set_ylabel('Best-hit identity (%)')
ax.legend(handles=[Line2D([],[],color=CAN,lw=1.5,label='canonical EAA2'),
                   Line2D([],[],color=SCA,lw=1.1,label='scaffold-retaining'),
                   Line2D([],[],color=CYT,lw=1.1,label='cytosolic')],
          frameon=False, loc='lower left', handlelength=1.4, borderpad=.1, labelspacing=.25)
plab(ax,'a',-.22)

ax = fig.add_subplot(gs[1]); tidy(ax)
t = table_6.sort_values('slope')
ax.barh(range(len(t)), t.slope, xerr=t.SE, color=[COLOR[q] for q in t.isoform], height=.66,
        error_kw=dict(lw=.75, capsize=1.8, ecolor='#333'), zorder=3)
ax.axvline(table_6.loc[table_6.isoform=='P43004','slope'].iloc[0], color=CAN, ls='--', lw=.8)
ax.set_yticks(range(len(t)))
ax.set_yticklabels([('$\\bf{'+q+'}$' if q=='P43004' else q) for q in t.isoform], fontsize=6.4)
ax.set_xlabel('Divergence slope\n(pp per log$_{10}$ Myr)'); ax.set_xlim(0,27)
ax.grid(axis='x', alpha=.18, lw=.5); ax.set_axisbelow(True); plab(ax,'b',-.55)

ax = fig.add_subplot(gs[2]); tidy(ax)
od = ['C9J9N5','A0A2R8Y4D1','A0A2R8Y4N0','A0A2R8Y642','A0A2R8YHI4','H0YEB1']
bp = ax.boxplot([(piv[q]-piv['P43004']).dropna().values for q in od], patch_artist=True,
        widths=.62, medianprops=dict(color='black',lw=.9), whiskerprops=dict(lw=.7),
        capprops=dict(lw=.7), flierprops=dict(ms=1.8, mec=GREY, mfc='none', mew=.5))
for b_, q in zip(bp['boxes'], od):
    b_.set_facecolor(COLOR[q]); b_.set_alpha(.8); b_.set_edgecolor('black'); b_.set_linewidth(.6)
ax.axhline(0, color=CAN, ls='--', lw=.9)
ax.set_xticklabels(od, rotation=45, ha='right', fontsize=6.2)
ax.set_ylabel('$\\Delta$ identity vs canonical (pp)')
ax.grid(axis='y', alpha=.18, lw=.5); ax.set_axisbelow(True); plab(ax,'c',-.34)
fig.savefig(f'{OUT}/Figure_4_molecular_clock.png', bbox_inches='tight', facecolor='white')
plt.close(); print('Figure 4 ->', f'{OUT}/Figure_4_molecular_clock.png')

In [ ]:
fig = plt.figure(figsize=(W_IN, 4.45))
gs = fig.add_gridspec(2,3, height_ratios=[1,1.1], hspace=.72, wspace=.44,
                      left=.085, right=.995, top=.90, bottom=.115)
ax = fig.add_subplot(gs[0,:]); tidy(ax)
gg = G['EAA2'].sort_values('pos')
sm_ = np.convolve(gg.score, np.ones(9)/9, mode='same')
for nm,(a,b_) in SCAFFOLD.items():
    ax.axvspan(a,b_, color=SCAF_C, alpha=.16, lw=0, zorder=0)
    ax.text((a+b_)/2, -1.72, nm, ha='center', fontsize=5.8, color='#2A5580', fontweight='bold')
for nm,(a,b_) in TRANSPORT.items():
    ax.axvspan(a,b_, color=TRAN_C, alpha=.16, lw=0, zorder=0)
    ax.text((a+b_)/2, -1.72, nm, ha='center', fontsize=5.8, color='#9C4A1E', fontweight='bold')
ax.fill_between(gg.pos, sm_, 0, where=sm_<0, color=SCA, alpha=.4, lw=0, zorder=2)
ax.plot(gg.pos, sm_, color='#1A1A1A', lw=.7, zorder=3)
ax.axhline(0, color=GREY, lw=.55, zorder=1)
ax.set_xlim(1,574); ax.set_ylim(-1.95,2.05); ax.invert_yaxis()
ax.set_xlabel('Residue position, canonical EAA2 (P43004)', labelpad=1.5)
ax.set_ylabel('ConSurf score\n($\\leftarrow$ more conserved)', labelpad=2, fontsize=7.6)
ax.legend(handles=[Patch(facecolor=SCAF_C, alpha=.35, label='scaffold / trimerisation domain'),
                   Patch(facecolor=TRAN_C, alpha=.35, label='transport domain')],
          frameon=False, ncol=2, loc='upper center', bbox_to_anchor=(.5,1.30), handlelength=1.3)
plab(ax,'a',-.082,1.30)

ax = fig.add_subplot(gs[1,0]); tidy(ax)
pv = ax.violinplot([sc_r.score, tr_r.score, oth.score], showmedians=True, widths=.78)
for pc, c_ in zip(pv['bodies'], [SCAF_C, TRAN_C, GREY]):
    pc.set_facecolor(c_); pc.set_alpha(.65); pc.set_edgecolor('black'); pc.set_linewidth(.5)
for kk in ('cbars','cmins','cmaxes','cmedians'):
    pv[kk].set_linewidth(.8); pv[kk].set_color('black')
ax.set_xticks([1,2,3]); ax.set_xticklabels(['scaffold','transport','loops'], fontsize=6.8)
ax.set_ylabel('ConSurf score'); ax.invert_yaxis()
ax.text(.5,1.02, fmt_p(p_dom), transform=ax.transAxes, ha='center', fontsize=6.6)
ax.grid(axis='y', alpha=.18, lw=.5); ax.set_axisbelow(True); plab(ax,'b',-.35)

ax = fig.add_subplot(gs[1,1]); tidy(ax)
ax.bar(range(len(table_5)), table_5.pearson_r,
       color=[COLOR[q] for q in table_5.isoform], width=.66, edgecolor='black', lw=.5, zorder=3)
for i,(r_,n_) in enumerate(zip(table_5.pearson_r, table_5.register_offset)):
    ax.text(i, r_+.025, f'+{n_}', ha='center', fontsize=5.6, color='#444')
ax.set_xticks(range(len(table_5)))
ax.set_xticklabels(table_5.isoform, rotation=45, ha='right', fontsize=6.2)
ax.set_ylabel('Pearson $r$ vs canonical'); ax.set_ylim(0,1.06)
ax.grid(axis='y', alpha=.18, lw=.5); ax.set_axisbelow(True); plab(ax,'c',-.35)

ax = fig.add_subplot(gs[1,2]); tidy(ax)
t9 = table_S9.iloc[::-1]
ax.barh(range(len(t9)), t9.difference, height=.64, edgecolor='black', lw=.5, zorder=3,
        color=[CAN if k=='EAA2' else SCA for k in t9.protein])
ax.set_yticks(range(len(t9)))
ax.set_yticklabels([('$\\bf{EAA2}$' if k=='EAA2' else k) for k in t9.protein], fontsize=6.2)
ax.set_xlabel('median(scaffold) $-$\nmedian(transport)', fontsize=7.4)
ax.axvline(0, color='black', lw=.65)
ax.grid(axis='x', alpha=.18, lw=.5); ax.set_axisbelow(True); plab(ax,'d',-.52)
fig.savefig(f'{OUT}/Figure_5_conservation_architecture.png', bbox_inches='tight', facecolor='white')
plt.close(); print('Figure 5 ->', f'{OUT}/Figure_5_conservation_architecture.png')

In [ ]:
genes_plot = [g_ for g_ in ['EAA1','EAA5'] if g_ in DENS]
fig, axes = plt.subplots(len(genes_plot), 1, figsize=(W_IN, 1.65*len(genes_plot)))
axes = np.atleast_1d(axes)
fig.subplots_adjust(left=.095, right=.995, top=.90, bottom=.13, hspace=.62)
META = {'EAA1':("SCI 1.678   ·   8 isoforms   ·   Tajima's $D$ = \u22121.100", '#16324F'),
        'EAA5':("SCI 1.500   ·   1 isoform   ·   Tajima's $D$ = +0.057", '#D9713C')}
for ax, gene, lab in zip(axes, genes_plot, ['a','b']):
    tidy(ax); dd = DENS[gene]
    xs = np.arange(len(dd['all']))*dd['W']; col = META[gene][1]
    ax.bar(xs, dd['all'], width=dd['W']*.9, color=GREY, alpha=.42, lw=0, zorder=2)
    ax.bar(xs, dd['comp'], width=dd['W']*.9, color=col, lw=0, zorder=3)
    ax.set_xlim(-60, dd['L']+60); ax.set_ylim(0,80)
    ax.set_ylabel('base pairs\nper 100 nt', fontsize=7.6)
    pct = 100*dd['comp'].sum()/max(dd['all'].sum(),1)
    ax.set_title(f'{gene}   —   {META[gene][0]}', loc='left', fontsize=8, pad=4)
    ax.text(.995,.80, f'{pct:.1f} % compensatory', transform=ax.transAxes, ha='right',
            fontsize=8, fontweight='bold', color=col)
    ax.grid(axis='y', alpha=.16, lw=.5); ax.set_axisbelow(True); plab(ax, lab, -.085, 1.12)
axes[0].legend(handles=[Patch(facecolor=GREY, alpha=.5, label='all supported base pairs'),
        Patch(facecolor='#16324F', label='compensatory (double) substitution')],
        frameon=False, ncol=2, loc='upper center', bbox_to_anchor=(.5,1.46), handlelength=1.3)
axes[-1].set_xlabel('Position in hominid consensus alignment (nt)')
fig.savefig(f'{OUT}/Figure_6_compensatory_landscape.png', bbox_inches='tight', facecolor='white')
plt.close(); print('Figure 6 ->', f'{OUT}/Figure_6_compensatory_landscape.png')

## 8. Audit of missing datasets

Three datasets referenced by the manuscript are not in the repository. This cell re-checks
for them, so if you deposit any of them the notebook will tell you the analysis can proceed.

The SpliceAI clustering analysis in particular is **blocked, not skipped** — the plotting
script reads a `data.dat` that was never committed.

In [ ]:
print('='*72)
dat = glob.glob(os.path.join(REPO,'**','*.dat'), recursive=True)
vcf = glob.glob(os.path.join(REPO,'**','*spliceai*'), recursive=True)
print(f'1. SpliceAI delta-score tables : {len(dat)} .dat file(s) found')
if dat: print('   ->', *dat, sep='\n      ')
else:
    print('   -> ABSENT. SpliceAI/codeforplot.gpl reads a data.dat that was never')
    print('      committed (5 columns: position, AL, DL, AG, DG). Without it the')
    print('      clustering analysis cannot be run on real scores.')
print()
missing_cov = [g_ for g_ in GENES if g_ not in COV]
print(f'2. RNAalifold pair lists       : missing for {missing_cov if missing_cov else "none"}')
if missing_cov:
    print('   -> re-run RNAalifold with pair output to complete Table 4.')

print()
qs = set(d['query'])
eaa1 = {'P43003','A0A7P0Z4R4','A0A7P0T9Z4','A0A087X0U3','A0A7P0T8Q1','E7EUV6','E7EUS7',
        'A0A7P0T9A4','A0A7P0T807'}
print(f'3. TBLASTN for EAA1 series     : {len(qs & eaa1)} of 9 EAA1 queries present')
if not (qs & eaa1):
    print('   -> ABSENT. Running the same 15 assemblies against P43003 and its eight')
    print('      isoforms would let Section 3 be replicated in a second gene.')
print('='*72)

## 9. Outputs

Everything written to `outputs/`:

In [ ]:
for f in sorted(os.listdir(OUT)):
    print(f'  {f:45s} {os.path.getsize(os.path.join(OUT,f)):>9,} bytes')
print()
print('Tables 4, 5, 6 and S5-S9 are CSV; Figures 4-6 are 600 dpi PNG at 6.8 in wide.')